# Error Case 23 — 500: MinIO Bucket Not Found

**Error:** `500 Internal Server Error`  
**Root cause:** Catalog points to non-existent MinIO bucket → storage location not accessible  
**Fix:** Verify bucket exists before catalog creation

## Flow
```
Catalog created with default-base-location: s3a://nonexistent-bucket/
  → STS credential vending succeeds (Polaris doesn't check bucket)
  → Table create → Spark tries to write to bucket
  → MinIO: bucket not found → S3 NoSuchBucket
  → Polaris or Spark: 500 / storage exception
```

## Why this happens in production
```
1. Typo in bucket name during catalog setup
2. Bucket deleted after catalog was created
3. Wrong environment (prod bucket name in dev)
4. MinIO bucket not created before catalog setup
```

In [ ]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD

cleanup()
print('✅ Setup complete')

In [ ]:
# Cell 2: Create catalog pointing to non-existent bucket
NONEXISTENT_BUCKET = 'nonexistent-bucket-xyz'

r = requests.post(f'{BASE_MGMT}/catalogs', headers=h(), json={
    'catalog': {
        'name': TEST_CATALOG,
        'type': 'INTERNAL',
        'properties': {
            'default-base-location': f's3a://{NONEXISTENT_BUCKET}/data/',
        },
        'storageConfigInfo': {
            'storageType': 'S3',
            'allowedLocations': [f's3a://{NONEXISTENT_BUCKET}/'],
            'pathStyleAccess': True,
            'endpoint': 'http://192.168.139.2:9000',
            'endpointInternal': 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000',
        },
    }
})
print_response(r, 'Create catalog with nonexistent bucket')
print(f'\nNote: Catalog create succeeds — Polaris does not verify bucket')
print(f'Bucket: {NONEXISTENT_BUCKET} (does not exist in MinIO)')

In [ ]:
# Cell 3: Create namespace (triggers STS + MinIO check)
if r.status_code == 201:
    r_ns = requests.post(
        f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
        headers=h(),
        json={'namespace': [TEST_NAMESPACE], 'properties': {}}
    )
    print_response(r_ns, 'Namespace create — triggers storage check')
    print(f'\nStatus: {r_ns.status_code}')
    if r_ns.status_code == 500:
        print('\n✅ 500 confirmed — bucket not accessible')
    elif r_ns.status_code in [200, 201]:
        print('\n⚠️  Namespace created — Polaris may not check bucket at namespace level')
        print('    Bucket check happens at table create or data write')
        
        # Try table create
        r_tbl = requests.post(
            f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}/tables',
            headers=h(),
            json={
                'name': TEST_TABLE,
                'schema': {
                    'type': 'struct',
                    'fields': [
                        {'id': 1, 'name': 'id', 'required': True, 'type': 'long'}
                    ]
                },
                'write-order': {'type': 'unsorted'},
                'stage-create': False
            }
        )
        print_response(r_tbl, 'Table create — triggers bucket check')
        print(f'Table create status: {r_tbl.status_code}')

In [ ]:
# Cell 4: Verify bucket existence in MinIO directly
import subprocess

result = subprocess.run([
    'kubectl', 'run', 'minio-check', '--rm', '-i',
    '--restart=Never',
    '--image=minio/mc:latest',
    '-n', 'datahub-hynix',
    '--',
    'sh', '-c',
    f'mc alias set minio http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000 minioadmin minioadmin && mc ls minio/{NONEXISTENT_BUCKET} 2>&1 || echo "BUCKET_NOT_FOUND"'
], capture_output=True, text=True, timeout=30)
print('MinIO bucket check:')
print(result.stdout or result.stderr)

if 'BUCKET_NOT_FOUND' in (result.stdout + result.stderr):
    print(f'\n✅ Confirmed: bucket {NONEXISTENT_BUCKET} does not exist')

In [ ]:
# Cell 5: Find logs in OpenSearch
wait_for_logs(10)
print('Searching for storage/bucket errors...')
hits = search_500_errors(minutes_ago=2)
if not hits:
    hits = search_db_errors(minutes_ago=2)
print_logs(hits)

In [ ]:
# Cell 6: Fix — verify bucket before catalog creation
def verify_bucket_exists(bucket_name):
    """Check bucket exists in MinIO before creating catalog."""
    import subprocess
    result = subprocess.run([
        'kubectl', 'run', 'bucket-verify', '--rm', '-i',
        '--restart=Never', '--image=minio/mc:latest',
        '-n', 'datahub-hynix', '--',
        'sh', '-c',
        f'mc alias set minio http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000 minioadmin minioadmin && mc ls minio/{bucket_name} > /dev/null 2>&1 && echo OK || echo MISSING'
    ], capture_output=True, text=True, timeout=30)
    return 'OK' in result.stdout

# Test with real bucket
print('Checking real bucket (data-catalog-bucket)...')
exists = verify_bucket_exists('data-catalog-bucket')
print(f'data-catalog-bucket exists: {exists}')

# Test with fake bucket
print(f'\nChecking fake bucket ({NONEXISTENT_BUCKET})...')
exists_fake = verify_bucket_exists(NONEXISTENT_BUCKET)
print(f'{NONEXISTENT_BUCKET} exists: {exists_fake}')

# Create catalog only if bucket exists
if verify_bucket_exists('data-catalog-bucket'):
    r_fix = create_catalog()   # uses real bucket
    print(f'\nCreate catalog with verified bucket: {r_fix.status_code}')
    assert r_fix.status_code == 201
    print('✅ Fixed — bucket verified before catalog creation')

cleanup()
print('\n=== Summary ===')
print('Error:    500 on table create or data write')
print('Cause:    Catalog bucket does not exist in MinIO')
print('Detect:   500 with S3/storage error after successful catalog create')
print('Tricky:   Catalog create succeeds — error appears later')
print('Fix:      Verify bucket exists before catalog setup')
print('Prevent:  Add bucket existence check to catalog creation script')